# Enmarcado del Problema (Problem Framing)
## Predicción del precio de diamantes a partir de sus características físicas

**Módulo 1 — Programación Analítica: Preparación de datos**

Este notebook responde, en formato consecutivo (pregunta → respuesta), a las preguntas de enmarcado del problema (*problem framing*) siguiendo el enfoque de la metodología de Aurélien Géron en *Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow* (capítulo 2, "End-to-End Machine Learning Project"), adaptadas al dataset de diamantes descrito en `data/01_raw/datos_diamantes_Info.txt`.

Fuente original del dataset (antes de la modificación usada en este curso): https://www.kaggle.com/shivam2503/diamonds — *nota: no se debe usar el dataset original, sino la versión modificada suministrada para el curso.*


## Pregunta 1: ¿Cuál es el objetivo del problema?

**Respuesta:**

El objetivo es construir un **modelo predictivo de regresión** que estime el precio en dólares (`price`) de un diamante a partir de sus características físicas y de calidad (`carat`, `cut`, `color`, `clarity`, `depth`, `table`, `x`, `y`, `z`).

Sin embargo, el objetivo del *negocio* va más allá de "predecir un número": en el mercado de diamantes existe una fuerte asimetría de información entre joyeros/tasadores y compradores/vendedores finales. Un sistema de este tipo normalmente se usa para:

- **Detectar sobreprecio o subprecio** en una pieza respecto al precio "justo" implícito en el mercado (útil para compradores, aseguradoras y casas de empeño).
- **Apoyar la tasación automática** en plataformas de compraventa (ej. joyerías en línea, casas de subastas como Blue Nile o James Allen), reduciendo el tiempo y costo de un tasador humano (gemólogo GIA) para cada pieza.
- **Servir como insumo (feature/score)** dentro de un sistema mayor, por ejemplo un motor de recomendación de compra o un sistema de detección de fraude en seguros de joyería.

Esta distinción importa porque cambia la métrica de éxito: no es lo mismo minimizar el error absoluto promedio que minimizar el error porcentual (importante cuando los precios van de \$326 a \$18,823, es decir, dos órdenes de magnitud de diferencia).

*Referencia:* Géron, A. (2019). *Hands-On Machine Learning*, Cap. 2 — sección "Look at the Big Picture": https://github.com/ageron/handson-ml3


## Pregunta 2: ¿Cómo se usará su solución?

**Respuesta:**

Dependiendo del contexto de despliegue, hay al menos tres formas típicas de uso, y es importante ser explícito sobre cuál se asume aquí:

1. **Herramienta de apoyo a la decisión (no automatizada al 100%):** el modelo entrega un precio estimado + un intervalo de confianza; un tasador humano lo usa como punto de referencia antes de dar el precio final. Este es el uso más realista y responsable dado que el precio de un diamante también depende de factores no capturados en los datos (certificación GIA/AGS, fluorescencia, procedencia ética/Kimberley Process, tendencias de mercado, marca del joyero).
2. **Componente de un pipeline mayor (batch):** el modelo se ejecuta periódicamente sobre un catálogo completo (ej. cada noche) para actualizar precios sugeridos en un e-commerce.
3. **Servicio en línea (API) para cotización instantánea:** un usuario ingresa las características (o las obtiene de un certificado GIA escaneado) y recibe un precio estimado en tiempo real.

Para este proyecto académico (Módulo 1, solo preparación de datos) se asume el escenario **(1) o (2) — modo *offline*/batch**, ya que no hay indicios de requerimientos de latencia en tiempo real ni de un flujo continuo de nuevos diamantes.

*Referencia sobre patrones de despliegue de modelos:* Google Cloud, "MLOps: Continuous delivery and automation pipelines in machine learning": https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning


## Pregunta 3: ¿Cuáles son las soluciones actuales (si las hay)?

**Respuesta:**

Actualmente, en la práctica de la industria joyera, la tasación de precios se resuelve de una de estas formas (que funcionan como *baseline* a superar):

- **Tasación manual por gemólogos certificados (GIA — Gemological Institute of America):** se basa en las "4 Cs" (Carat, Cut, Color, Clarity) más el "Rapaport Diamond Report" (*Rapaport Price List*), una lista de precios de referencia semanal usada globalmente por la industria como *benchmark* de precio por quilate según las 4Cs. Es la "solución actual" de facto en la industria: https://www.diamonds.net/prices/
- **Modelos hedónicos de precios (hedonic pricing regression)** usados en estudios econométricos de joyería desde los años 80-90, que son básicamente regresiones lineales múltiples con las mismas variables (peso, color, claridad, corte) — el problema que se está resolviendo aquí es, en esencia, una versión moderna (ML) de estos modelos hedónicos clásicos.
- **Kernels públicos de Kaggle** sobre el dataset original de diamantes, que suelen usar regresión lineal, Random Forest, XGBoost y logran típicamente R² > 0.97 y RMSE relativamente bajo respecto al rango de precios — esto da una cota de referencia de qué tan "resoluble" es el problema con este tipo de variables.

*Referencias:*
- Rapaport Price List: https://www.diamonds.net/prices/
- Kaggle — "Diamonds" dataset y notebooks de la comunidad: https://www.kaggle.com/datasets/shivam2503/diamonds/code


## Pregunta 4: ¿Cómo se debe enmarcar este problema (supervisado / no supervisado, en línea / fuera de línea, etc.)?

**Respuesta:**

| Dimensión | Clasificación | Justificación |
|---|---|---|
| Tipo de aprendizaje | **Supervisado** | Se dispone de la variable objetivo `price` (etiqueta) para cada observación histórica. |
| Tipo de tarea | **Regresión** (no clasificación) | `price` es una variable continua ($326–$18,823), no una categoría. |
| Modo de entrenamiento | **Batch / offline learning** | No hay evidencia de un flujo continuo de datos nuevos ni restricciones de memoria que obliguen a *online learning* (aprendizaje incremental tipo `partial_fit`). El dataset es estático y cabe en memoria. |
| Modo de predicción (inferencia) | Puede ser **batch** (tasación masiva de catálogo) o **online/on-demand** (una cotización a la vez vía API) — ambos son compatibles con un modelo entrenado en batch. | El *entrenamiento* offline no impide *servir* el modelo en tiempo real después. |
| Instance-based vs Model-based | **Model-based** (se busca generalizar mediante una función paramétrica, ej. regresión lineal/regularizada, árboles de decisión, ensambles) — aunque también podría explorarse un enfoque *instance-based* como k-NN por similitud de diamantes. | Dado el volumen de datos y la necesidad de generalizar, el enfoque model-based suele ser más práctico y interpretable. |
| Multivariante | **Sí, multivariante multivariable** | 8 variables predictoras (carat, cut, color, clarity, depth, table, x, y, z) de tipos mixtos (numéricas continuas + categóricas ordinales). |

Un matiz importante: `cut`, `color` y `clarity` son **variables categóricas ordinales** (tienen un orden natural de calidad), no nominales. Esto debe reflejarse en la preparación de datos (usar `OrdinalEncoder` con el orden correcto, no un One-Hot Encoding "ciego" que ignore el orden, aunque One-Hot también es defendible según el algoritmo elegido).

*Referencia:* Géron, A., *Hands-On ML*, Cap. 1 — taxonomía de sistemas de ML (supervised/unsupervised, batch/online, instance/model-based): https://github.com/ageron/handson-ml3


## Pregunta 5: ¿Cómo se debe medir el desempeño o el rendimiento de la solución, una primera intuición?

**Respuesta:**

Para una tarea de regresión, las métricas estándar de primera intuición son:

- **RMSE (Root Mean Squared Error):** penaliza más los errores grandes (cuadráticamente). Es sensible a *outliers*, lo cual es relevante aquí porque hay diamantes muy caros (hasta \$18,823) que, si se predicen mal, generan errores absolutos enormes.
- **MAE (Mean Absolute Error):** más robusta a outliers, más fácil de interpretar ("en promedio nos equivocamos en $X").
- **MAPE (Mean Absolute Percentage Error) o RMSLE (Root Mean Squared Log Error):** dado que el precio abarca dos órdenes de magnitud (\$326 a \$18,823), un error de \$500 es trivial en un diamante de \$15,000 pero catastrófico en uno de \$400. Por eso, en datasets de precios con esta dispersión, suele preferirse **RMSLE** o entrenar sobre `log(price)` y evaluar en escala logarítmica — esta es una intuición *no trivial* pero estándar en Kaggle para este tipo de datasets (de hecho, muchas competencias de precios, como "House Prices" de Kaggle, usan explícitamente RMSLE por esta misma razón).
- **R² (coeficiente de determinación):** útil como métrica complementaria de "cuánta varianza explica el modelo", buena para comunicar resultados a no-técnicos, pero no debe ser la única métrica de optimización.

**Intuición inicial recomendada:** usar **RMSE sobre `log(price)`** (equivalente a RMSLE) como métrica principal de optimización, y reportar MAE y R² en escala original como métricas de interpretación para el negocio.

*Referencias:*
- Scikit-learn, "Regression metrics": https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics
- Kaggle, "House Prices - Advanced Regression Techniques" (ejemplo de uso de RMSLE en un problema de precios con distribución sesgada, análogo a este caso): https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques


## Pregunta 6: ¿La medida de desempeño está alineada con el objetivo del problema?

**Respuesta:**

Parcialmente, y es importante ser crítico aquí:

- **RMSE/MAE en dólares absolutos** están alineadas si el objetivo de negocio es "minimizar la pérdida monetaria promedio por tasación incorrecta" (ej. una aseguradora que paga de más). Pero **no** están bien alineadas si el objetivo es "ser justo en términos relativos" con diamantes de todos los rangos de precio, porque el error se concentra (por construcción del RMSE) en los diamantes más caros, que son minoría en el dataset.
- **RMSLE / entrenar en log-price** está mejor alineado con un objetivo de "equidad relativa" (tratar con la misma importancia un error del 10% en un diamante barato que en uno caro), lo cual suele ser más justo para un sistema que se usará sobre *todo* el catálogo, no solo sobre piezas de alto valor.
- Ninguna de estas métricas captura directamente el **costo asimétrico** de sobreestimar vs. subestimar el precio (por ejemplo, en un sistema de compra, subestimar el precio de venta le cuesta dinero al vendedor; sobreestimarlo puede espantar al comprador). Si el objetivo real del negocio tiene esa asimetría, se necesitaría una métrica custom (ej. una función de pérdida asimétrica), algo que no se resuelve con RMSE/MAE estándar.

**Conclusión:** la métrica propuesta (RMSE sobre log-price) está razonablemente alineada con el objetivo de tasación general, pero se recomienda validarla con el "dueño del negocio" antes de fijarla como métrica final, especialmente en cuanto a la asimetría de costos.

*Referencia:* Provost, F. & Fawcett, T. (2013). *Data Science for Business* — Cap. 7, sobre alineación entre métricas de evaluación y objetivos de negocio (costo-beneficio asimétrico).


## Pregunta 7: ¿Cuál sería el desempeño o rendimiento mínimo necesario para alcanzar el objetivo del problema?

**Respuesta:**

No hay un umbral "mágico" universal; se debe derivar comparando contra el *baseline* actual (pregunta 3):

- Si el objetivo es **reemplazar o apoyar la tasación del Rapaport Price List / gemólogo humano**, el modelo debería lograr un error porcentual medio (MAPE) **inferior al margen de negociación típico del mercado de diamantes**, que suele rondar el 5–10% sobre el precio de lista. Es decir, un **MAPE objetivo < 10%** sería un mínimo razonable para que el modelo sea "útil" y no solo "interesante".
- En términos de R², los notebooks públicos de Kaggle sobre este mismo dataset (con Random Forest / Gradient Boosting) suelen alcanzar **R² ≥ 0.97–0.98**. Por tanto, un mínimo razonable de aceptación para este proyecto sería **R² ≥ 0.90** como piso "aceptable" para un primer modelo (Módulo 1 es solo preparación de datos, así que este umbral aplica a fases posteriores del proyecto), y R² ≥ 0.97 como meta "competitiva" frente al estado del arte.
- Un mínimo *no negociable*: el modelo debe superar de forma significativa un **baseline trivial** (ej. predecir siempre la media o mediana del precio, o una regresión lineal simple usando solo `carat`, que ya por sí sola correlaciona fuertemente con `price`). Si el modelo final no mejora sustancialmente ese baseline, no se justifica la complejidad adicional.

*Referencia:* discusión de baselines triviales como piso de comparación en Géron, A., *Hands-On ML*, Cap. 2, sección "Select a Performance Measure" y "sanity check" con `DummyRegressor` de scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html


## Pregunta 8: ¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencia o herramientas ya creadas?

**Respuesta:**

Este problema pertenece a la familia de **regresión de precios de bienes heterogéneos (hedonic pricing / price prediction)**, muy estudiada en ML. Problemas análogos de los que se puede reutilizar experiencia, código y arquitectura:

- **Predicción de precios de vivienda** (ej. "California Housing" — el mismo dataset usado en Géron, *Hands-On ML*, Cap. 2 — o "House Prices: Advanced Regression Techniques" de Kaggle): comparte la mezcla de variables numéricas y categóricas ordinales/nominales, la necesidad de manejar outliers de precio, y el uso de pipelines de `Pipeline` + `ColumnTransformer` de scikit-learn. La estructura completa del pipeline de preparación de datos (imputación, escalado, codificación) es directamente reutilizable.
- **Predicción de precios de autos usados** (ej. datasets de Kaggle "Used Car Price Prediction"): comparte variables categóricas ordinales de "condición/calidad" análogas a `cut`/`clarity`/`color`.
- **Predicción de precios de vinos** ("Wine Quality dataset" de UCI): estructura muy similar — variables físico-químicas continuas prediciendo un score/precio, buen paralelo metodológico.

En términos de **herramientas reutilizables**: scikit-learn (`Pipeline`, `ColumnTransformer`, `OrdinalEncoder`, `StandardScaler`), y algoritmos de ensamble (`RandomForestRegressor`, `GradientBoostingRegressor`, `XGBoost`, `LightGBM`) que consistentemente funcionan bien en este tipo de problemas tabulares con relaciones no lineales.

*Referencias:*
- California Housing dataset (Géron, Cap. 2): https://github.com/ageron/handson-ml3/blob/main/02_end_to_end_machine_learning_project.ipynb
- UCI Wine Quality dataset: https://archive.ics.uci.edu/dataset/186/wine+quality


## Pregunta 9: ¿Hay experiencia del problema disponible?

**Respuesta:**

Sí, en varios niveles:

1. **Experiencia de dominio (gemología):** existe amplia literatura y estándares de la industria sobre cómo las 4Cs determinan el precio — el estándar GIA (Gemological Institute of America) documenta con precisión cómo `carat`, `cut`, `color` y `clarity` interactúan de forma no lineal en el precio (por ejemplo, hay "saltos" de precio en quilates "redondos" como 1.00 ct vs 0.99 ct, un efecto psicológico de mercado bien documentado). Esta experiencia de dominio es clave para el *feature engineering* posterior (Módulo 2+).
2. **Experiencia técnica previa (Kaggle):** el dataset original de diamantes (fuente de este dataset modificado) es uno de los datasets más usados en la comunidad de Kaggle para practicar regresión, con cientos de notebooks públicos, discusiones y soluciones de referencia disponibles.
3. **Experiencia académica:** existen papers de economía y gemología sobre "hedonic pricing of diamonds" desde los años 80 que ya identificaron estas mismas variables como determinantes del precio, mucho antes de que existiera el machine learning moderno.

*Referencias:*
- GIA — "The 4Cs of Diamond Quality": https://www.gia.edu/4cs-diamond-quality
- Kaggle — notebooks de la comunidad sobre el dataset de diamantes: https://www.kaggle.com/datasets/shivam2503/diamonds/code


## Pregunta 10 (Importante): ¿Cómo se puede resolver el problema manualmente?

**Respuesta:**

Un experto humano (gemólogo o joyero tasador) resolvería el problema manualmente siguiendo, aproximadamente, este procedimiento:

1. **Determinar el precio base por quilate** consultando una lista de referencia como el **Rapaport Diamond Report**, que publica semanalmente una matriz de precios por quilate según combinaciones de color y claridad, segmentada por rangos de peso (ej. 0.90–0.99 ct, 1.00–1.49 ct, etc.).
2. **Aplicar un ajuste porcentual (descuento o prima) sobre ese precio base** según:
   - La calidad del corte (`cut`): un corte "Ideal" puede tener una prima de +5% a +10% sobre un corte "Good", porque afecta directamente el brillo ("fire" y "brilliance") del diamante.
   - Proporciones específicas (`depth`, `table`, y las dimensiones `x`, `y`, `z`): un tasador experto revisa si el `depth%` y `table%` caen dentro de los rangos "ideales" conocidos (ej. depth ~59-62.3%, table ~54-58% para un corte redondo brillante) — desviarse de esos rangos reduce el valor aunque el corte esté catalogado como "Ideal".
   - Factores no incluidos en este dataset pero usados en la práctica real: fluorescencia, simetría, pulido, forma (redondo, princesa, esmeralda...), y certificación (GIA vs. otros laboratorios menos reconocidos).
3. **Multiplicar** el precio ajustado por quilate × el peso en quilates (`carat`) para obtener el precio final.
4. **Ajustar por condiciones de mercado** (oferta/demanda del momento, tipo de comprador, negociación).

Este procedimiento manual es, en esencia, un **modelo lineal segmentado y basado en reglas de negocio** — es una buena referencia mental (baseline conceptual) de qué tan complejas son las interacciones que el modelo de ML deberá aprender a capturar automáticamente a partir de los datos.

*Referencia:* Rapaport Diamond Report — metodología de tasación: https://www.diamonds.net/prices/ ; GIA — "4Cs": https://www.gia.edu/4cs-diamond-quality


## Pregunta 11: Listado de los supuestos que hay hasta este momento

**Respuesta:**

1. Se asume que el dataset modificado suministrado (`datos_diamantes_Info.txt` + el archivo de datos asociado) **conserva la relación real** entre las variables físicas y el precio, y que la modificación respecto al dataset original de Kaggle no altera de forma sustancial esa relación (solo cambia, por ejemplo, formato, columnas, ruido añadido, o subconjunto de filas).
2. Se asume que **todos los diamantes del dataset corresponden a un mismo tipo de talla/forma (probablemente "round brilliant")**, ya que no hay una columna `shape`/`cut_shape` — de no ser así, comparar precios sin esa variable sería problemático porque la forma afecta fuertemente el precio.
3. Se asume que **los precios están en la misma moneda y unidad de tiempo** (USD, sin ajuste por inflación entre observaciones) — es decir, que el dataset no mezcla precios de años muy distintos sin normalizar.
4. Se asume que las variables `cut`, `color` y `clarity` siguen los **estándares GIA estándar de la industria** (y no una escala propietaria distinta de algún joyero específico).
5. Se asume que **no hay fuga de información (data leakage)** entre `x`, `y`, `z`, `depth` y `table` — de hecho, hay que verificar en Preparación de Datos si `depth` es consistente con la fórmula dada (`depth = 2z / (x+y)`), lo que podría indicar redundancia/colinealidad fuerte entre variables.
6. Se asume que el dataset **no contiene diamantes fraudulentos, sintéticos o tratados** (ej. HPHT, irradiados) sin etiquetar como tal, lo cual alteraría el precio esperado para las mismas características físicas.
7. Se asume, para efectos del Módulo 1, que **no hay restricciones de actualización en tiempo real** (ver pregunta 13) y que el dataset es una fotografía estática válida para entrenar y evaluar el modelo.
8. Se asume que los valores extremos en `x`, `y`, `z` iguales a 0 (mencionados en el rango, ej. `y: 0--58.9`) son **errores de captura de datos** y no diamantes reales de dimensión cero — esto deberá tratarse explícitamente en la limpieza de datos.


## Pregunta 12: ¿Cuál es la fuente de los datos?

**Respuesta:**

Según el archivo de descripción, la fuente es una **versión modificada** de un dataset público originalmente alojado en Kaggle:

> https://www.kaggle.com/shivam2503/diamonds

Ese dataset original de Kaggle, a su vez, es una compilación ampliamente utilizada derivada de datos de precios de venta de diamantes al detalle (comúnmente atribuida a datos recopilados de listados de diamantes de joyerías/proveedores, similar en espíritu a los datos que también sustentan el paquete `ggplot2::diamonds` de R, un dataset de referencia muy usado en estadística y visualización con ~54,000 diamantes).

**Nota importante explícita en el enunciado:** *"No usar el dataset original"* — es decir, para este curso se debe trabajar exclusivamente con el archivo modificado suministrado por el profesor/plataforma del curso, no con el CSV original descargado directamente de Kaggle. Esto probablemente se debe a que el dataset del curso incluye modificaciones deliberadas (valores faltantes, errores, duplicados, outliers introducidos) diseñadas específicamente para la práctica de limpieza y preparación de datos.

*Referencia adicional (dataset hermano ampliamente documentado):* documentación del dataset `diamonds` de R (mismo origen conceptual): https://ggplot2.tidyverse.org/reference/diamonds.html


## Pregunta 13: ¿Cómo se actualizan los datos?

**Respuesta:**

No hay evidencia en el archivo de descripción de que exista un **mecanismo de actualización automática** de este dataset — es un archivo estático (probablemente CSV) entregado una sola vez para el curso.

Sin embargo, es útil distinguir dos escenarios:

- **Escenario académico actual (el de este proyecto):** el dataset es una **instantánea fija (snapshot)**, descargada y modificada manualmente por el equipo docente. No se actualiza; cualquier "actualización" implicaría recibir un nuevo archivo del curso.
- **Escenario de producción real (si este sistema se llevara a producción):** en la industria, los datos de precios de diamantes sí se actualizarían mediante:
  - Ingesta periódica de nuevas transacciones/listados desde plataformas de venta (ej. APIs de casas de subasta o marketplaces de joyería).
  - Actualización semanal del Rapaport Price List como referencia de mercado.
  - Procesos ETL que consolidan certificados GIA nuevos con sus respectivos precios de venta reales.

Para efectos de este Módulo 1, se debe **documentar explícitamente que el dataset se trata como estático**, y cualquier pipeline de preparación de datos debe diseñarse de forma reproducible (ej. con funciones/`Pipeline` de scikit-learn) para que, si en el futuro llega una versión actualizada del dataset, el mismo proceso de limpieza pueda re-ejecutarse sin reescribir código desde cero.


## Pregunta 14: ¿Cada cuánto tiempo se actualizan los datos?

**Respuesta:**

Dado que, como se estableció en la pregunta anterior, el dataset de este curso es una **entrega estática única** (no hay flujo de actualización), la frecuencia de actualización aplicable a este proyecto es: **nunca / no aplica (dataset fijo para todo el módulo/curso)**.

Como contraste de referencia para un sistema real (para tener una intuición de qué tan "viejo" se volvería un modelo entrenado sobre datos de precios de diamantes):

- El **Rapaport Price List**, el benchmark de precios más usado en la industria, se actualiza **semanalmente**.
- En un sistema de producción real, sería razonable reentrenar o al menos **re-evaluar el modelo (model monitoring / drift detection) mensualmente o trimestralmente**, dado que los precios de commodities de lujo como los diamantes no cambian tan rápido día a día, pero sí pueden tener *drift* relevante por: cambios en tasas de cambio (USD), tendencias de moda (ej. auge de diamantes cultivados en laboratorio presionando a la baja el precio de diamantes naturales desde ~2019-2020), o cambios macroeconómicos.

Esta reflexión es relevante para el diseño del pipeline: aunque el dataset actual no se actualiza, un buen diseño de preparación de datos (Módulo 1) debe ser **reutilizable/parametrizable**, anticipando que en un escenario real habría que re-ejecutar el mismo proceso sobre datos futuros.

*Referencia:* sobre el impacto de los diamantes cultivados en laboratorio en el mercado de precios de diamantes naturales: https://www.bain.com/insights/global-diamond-industry-report/
